# Phase 3 – Rhetorical Role Prediction
## Experiment 2: LORMA Fine-Tuning on Architecture 2 (RoPE + SwiGLU + RMSNorm)

**Technique**: LORMA (Low-Rank Matrix Adaptation) — **COMPULSORY**  
**Base Model**: Phase-1 Architecture 2 – `RoPESwiGLUDecoderLM` (RoPE, SwiGLU, RMSNorm, QK-Norm, ~105M params)  
**Classification Head**: Transformer cross-attention head over sentence embeddings  
**Context Strategy**: Full document context via cross-attention  
**Domains**: CL + IT combined with domain token

---

### What is LORMA?
LORMA extends LoRA by applying low-rank adaptation to **both attention and feed-forward layers** simultaneously, with a separate rank budget per layer type. It uses:
- Rank-r adaptation on Q, K, V, O (attention)
- Rank-r' adaptation on gate_proj, up_proj (FFN)
- Tied scaling across layers within a block

This gives a more thorough adaptation compared to LoRA (attention-only) while remaining highly parameter-efficient.

## 1. Setup & Imports

In [ ]:
import os, json, math, random, time, zipfile, warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('Device:', device)

## 2. Configuration

In [ ]:
# Paths
PHASE1_MODEL_PATH = '/kaggle/input/phase1-arch2/decoder_lm.pt'
TOKENIZER_PATH    = '/kaggle/input/phase1-arch2/custom_bpe.json'
TRAIN_CL = '/kaggle/input/phase3-rr-data/Train/CL_train.json'
DEV_CL   = '/kaggle/input/phase3-rr-data/Train/CL_dev.json'
TRAIN_IT = '/kaggle/input/phase3-rr-data/Train/IT_train.json'
DEV_IT   = '/kaggle/input/phase3-rr-data/Train/IT_dev.json'
LABEL_VOCAB_PATH = '/kaggle/input/phase3-rr-data/Train/label_vocab.json'
TEST_CL  = '/kaggle/input/phase3-rr-data/Test/CL_test.json'
TEST_IT  = '/kaggle/input/phase3-rr-data/Test/IT_test.json'
OUTPUT_DIR = '/kaggle/working/phase3_exp2_lorma_arch2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# LORMA hyperparams
LORMA_RANK_ATTN  = 16   # rank for attention layers
LORMA_RANK_FFN   = 8    # rank for FFN layers (lower budget)
LORMA_ALPHA      = 32
LORMA_DROPOUT    = 0.05

# Training
MAX_SEQ_LEN    = 256
CONTEXT_WINDOW = 3      # wider context for arch2
BATCH_SIZE     = 16
LEARNING_RATE  = 3e-4
WEIGHT_DECAY   = 0.01
EPOCHS         = 10
GRAD_CLIP      = 1.0
DOMAIN_TOKEN   = True   # prepend [CL] or [IT] domain token

print('Experiment 2: LORMA on Architecture 2 configured.')

## 3. Tokenizer

In [ ]:
SPECIALS = ['<pad>', '<unk>', '<s>', '</s>']
END_OF_WORD = '</w>'

class ByteBPETokenizer:
    def __init__(self, vocab_size=16000):
        self.vocab_size = vocab_size
        self.merges = []; self.merge_rank = {}; self.itos = []; self.stoi = {}

    @staticmethod
    def _word_to_symbols(word):
        return list(word.encode('utf-8').decode('latin-1')) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        symbols = self._word_to_symbols(word)
        while len(symbols) > 1:
            pairs = [(symbols[i], symbols[i+1]) for i in range(len(symbols)-1)]
            cands = [(self.merge_rank[p], i) for i, p in enumerate(pairs) if p in self.merge_rank]
            if not cands: break
            _, idx = min(cands)
            symbols = symbols[:idx] + [symbols[idx]+symbols[idx+1]] + symbols[idx+2:]
        return symbols

    def encode(self, text, max_len=None):
        ids = [self.stoi.get(s, self.stoi['<unk>']) for w in text.split() for s in self._apply_merges_to_word(w)]
        return ids[:max_len] if max_len else ids

    def decode(self, ids):
        syms = [self.itos[i] if i < len(self.itos) else '' for i in ids]
        words = [w for w in ''.join(syms).split(END_OF_WORD) if w]
        return ' '.join(w.encode('latin-1').decode('utf-8', errors='replace') for w in words)

    @classmethod
    def load(cls, path):
        with open(path, 'r', encoding='utf-8') as f: data = json.load(f)
        tok = cls(vocab_size=data['vocab_size'])
        tok.itos = data['itos']
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.merges = [tuple(p) for p in data['merges']]
        tok.merge_rank = {p: i for i, p in enumerate(tok.merges)}
        return tok

tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
PAD_ID = tokenizer.stoi.get('<pad>', 0)
print(f'Tokenizer loaded. Vocab: {tokenizer.vocab_size}')

## 4. Load Data & Labels

In [ ]:
def load_json(p):
    with open(p) as f: return json.load(f)

with open(LABEL_VOCAB_PATH) as f: lv = json.load(f)
LABELS = lv if isinstance(lv, list) else list(lv.keys())
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL  = {i: l for l, i in LABEL2ID.items()}
NUM_LABELS = len(LABELS)
print('Labels:', LABELS)

cl_train = load_json(TRAIN_CL); cl_dev = load_json(DEV_CL)
it_train = load_json(TRAIN_IT); it_dev = load_json(DEV_IT)
cl_test  = load_json(TEST_CL);  it_test= load_json(TEST_IT)

print(f'CL: train={len(cl_train)}, dev={len(cl_dev)}, test={len(cl_test)}')
print(f'IT: train={len(it_train)}, dev={len(it_dev)}, test={len(it_test)}')

# Class weights
all_labels_train = [l for doc in cl_train+it_train for l in doc.get('labels',[])]
label_counts = Counter(all_labels_train)
total = sum(label_counts.values())
class_weights = torch.tensor(
    [total/(NUM_LABELS*label_counts.get(l,1)) for l in LABELS],
    dtype=torch.float32
).to(device)
print('Class weights computed.')

## 5. Architecture 2 Model Definition

In [ ]:
@dataclass
class Arch2Config:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    d_ff: int = 2560
    max_seq_len: int = 512
    dropout: float = 0.1
    rope_theta: float = 10000.0

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer('cos_cached', emb.cos(), persistent=False)
        self.register_buffer('sin_cached', emb.sin(), persistent=False)
    def forward(self, x, seq_len):
        return self.cos_cached[:seq_len].to(x.dtype), self.sin_cached[:seq_len].to(x.dtype)

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(1); sin = sin.unsqueeze(0).unsqueeze(1)
    return (q*cos)+(rotate_half(q)*sin), (k*cos)+(rotate_half(k)*sin)

class RoPECausalAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        self.dropout_p = cfg.dropout
        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        k = self.k_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        v = self.v_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        q = self.q_norm(q); k = self.k_norm(k)
        q, k = apply_rotary(q, k, cos, sin)
        z = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                dropout_p=self.dropout_p if self.training else 0.0)
        return self.resid_dropout(self.out_proj(z.transpose(1,2).contiguous().view(B,T,C)))

class SwiGLUFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up_proj   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.dropout   = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class RoPEBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn  = RoPECausalAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn   = SwiGLUFeedForward(cfg)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ffn(self.norm2(x))
        return x

class RoPESwiGLUDecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.rope = RotaryEmbedding(cfg.d_model // cfg.n_heads, cfg.max_seq_len, cfg.rope_theta)
        self.blocks = nn.ModuleList([RoPEBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos, sin = self.rope(x, T)
        for block in self.blocks:
            x = block(x, cos, sin)
        return self.norm_f(x)  # return hidden states

print('Architecture 2 (RoPE + SwiGLU + RMSNorm) defined.')

## 6. LORMA Implementation

In [ ]:
class LORMALinear(nn.Module):
    """
    LORMA: Low-Rank Matrix Adaptation.
    Applied to both attention AND FFN layers with separate rank budgets.
    
    Key difference from LoRA:
    - Uses weight decomposition W = W0 + s * B @ A where s is a learned scalar per layer
    - Separate rank for attention vs FFN layers
    - Rank-stabilized scaling: uses sqrt(rank) in denominator
    """
    def __init__(self, original: nn.Linear, rank: int, alpha: float, dropout: float = 0.0):
        super().__init__()
        self.original = original
        self.rank = rank
        self.alpha = alpha
        # Rank-stabilized scaling (rsLoRA style within LORMA)
        self.scaling = alpha / math.sqrt(rank)

        in_f  = original.in_features
        out_f = original.out_features

        self.lorma_A = nn.Linear(in_f, rank, bias=False)
        self.lorma_B = nn.Linear(rank, out_f, bias=False)
        # Learned per-layer scalar (initialized to 1)
        self.lorma_s = nn.Parameter(torch.ones(1))
        self.lorma_dropout = nn.Dropout(dropout)

        nn.init.kaiming_uniform_(self.lorma_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B.weight)

        # Freeze base weights
        self.original.weight.requires_grad = False
        if self.original.bias is not None:
            self.original.bias.requires_grad = False

    def forward(self, x):
        return self.original(x) + self.lorma_s * self.scaling * self.lorma_B(self.lorma_A(self.lorma_dropout(x)))

def inject_lorma(model: nn.Module, attn_modules: List[str], ffn_modules: List[str],
                  rank_attn: int, rank_ffn: int, alpha: float, dropout: float):
    """Inject LORMA into both attention and FFN projection layers."""
    replaced = {'attn': 0, 'ffn': 0}
    for name, module in list(model.named_modules()):
        for target in attn_modules:
            if name.endswith(target) and isinstance(module, nn.Linear):
                parts = name.split('.')
                parent = model
                for p in parts[:-1]: parent = getattr(parent, p)
                setattr(parent, parts[-1], LORMALinear(module, rank_attn, alpha, dropout))
                replaced['attn'] += 1
                break
        for target in ffn_modules:
            if name.endswith(target) and isinstance(module, nn.Linear):
                parts = name.split('.')
                parent = model
                for p in parts[:-1]: parent = getattr(parent, p)
                setattr(parent, parts[-1], LORMALinear(module, rank_ffn, alpha, dropout))
                replaced['ffn'] += 1
                break
    print(f'LORMA injected: {replaced["attn"]} attn layers (r={rank_attn}), '
          f'{replaced["ffn"]} ffn layers (r={rank_ffn})')
    return replaced

def count_trainable(model):
    tot = sum(p.numel() for p in model.parameters())
    tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Params: total={tot:,} | trainable={tr:,} ({100*tr/tot:.2f}%)')
    return tr

print('LORMA implementation ready.')

## 7. RR Classifier with Transformer Sentence Head

In [ ]:
class TransformerSentenceHead(nn.Module):
    """
    A small Transformer encoder that processes the sequence of sentence embeddings
    within a document for document-level context modeling.
    """
    def __init__(self, d_model: int, n_heads: int = 4, n_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*2,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [1, S, d_model] -> [S, d_model]"""
        return self.norm(self.encoder(x)).squeeze(0)

class RRClassifierArch2(nn.Module):
    """
    RR Classifier on Architecture 2.
    Pipeline: LORMA-adapted RoPESwiGLU backbone -> mean-pool -> Transformer head -> classifier
    Also uses domain token embedding to condition on CL vs IT.
    """
    def __init__(self, backbone: RoPESwiGLUDecoderLM, num_labels: int,
                 num_domains: int = 2, dropout: float = 0.1):
        super().__init__()
        self.backbone = backbone
        d_model = backbone.cfg.d_model
        self.num_labels = num_labels

        # Domain embedding (0=CL, 1=IT)
        self.domain_emb = nn.Embedding(num_domains, d_model)

        # Transformer sentence head
        self.sent_transformer = TransformerSentenceHead(d_model, n_heads=4, n_layers=2, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_labels)
        nn.init.xavier_uniform_(self.classifier.weight)

    def encode_sentences(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """Encode batch of sentences -> [S, d_model]"""
        hidden = self.backbone(input_ids)             # [S, T, d_model]
        mask_exp = attention_mask.unsqueeze(-1).float()
        sent_embs = (hidden * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)
        return sent_embs

    def forward(self, input_ids, attention_mask, domain_id: int = 0):
        """
        input_ids:      [S, T]
        attention_mask: [S, T]
        domain_id:      int (0=CL, 1=IT)
        Returns logits: [S, num_labels]
        """
        sent_embs = self.encode_sentences(input_ids, attention_mask)  # [S, d_model]

        # Add domain conditioning
        dom = self.domain_emb(torch.tensor(domain_id, device=input_ids.device))  # [d_model]
        sent_embs = sent_embs + dom.unsqueeze(0)  # [S, d_model]
        sent_embs = self.dropout(sent_embs)

        # Transformer over sentence sequence
        sent_embs = sent_embs.unsqueeze(0)        # [1, S, d_model]
        ctx_embs  = self.sent_transformer(sent_embs)  # [S, d_model]
        ctx_embs  = self.dropout(ctx_embs)

        return self.classifier(ctx_embs)           # [S, num_labels]

print('RRClassifierArch2 (Transformer head + domain token) defined.')

## 8. Load Phase 1 Checkpoint & Apply LORMA

In [ ]:
print('Loading Architecture 2 checkpoint...')
ckpt = torch.load(PHASE1_MODEL_PATH, map_location='cpu')

# Reconstruct Arch2Config
raw = ckpt.get('config', {})
if hasattr(raw, 'd_model'):  # dataclass
    cfg = raw
else:
    if hasattr(raw, '__dict__'): raw = raw.__dict__
    d = raw if isinstance(raw, dict) else {}
    cfg = Arch2Config(
        vocab_size=ckpt.get('vocab_size', tokenizer.vocab_size),
        d_model=d.get('d_model', 768), n_layers=d.get('n_layers', 12),
        n_heads=d.get('n_heads', 12), d_ff=d.get('d_ff', 2560),
        max_seq_len=d.get('max_seq_len', 512)
    )
print(f'Arch2 Config: d_model={cfg.d_model}, n_layers={cfg.n_layers}, d_ff={cfg.d_ff}')

backbone = RoPESwiGLUDecoderLM(cfg)
backbone.load_state_dict(ckpt['model_state_dict'], strict=False)
print('Backbone weights loaded.')

# Freeze all backbone params
for p in backbone.parameters():
    p.requires_grad = False

# Inject LORMA into attention AND FFN layers
ATTN_MODULES = ['q_proj', 'k_proj', 'v_proj', 'out_proj']
FFN_MODULES  = ['gate_proj', 'up_proj', 'down_proj']
inject_lorma(backbone, ATTN_MODULES, FFN_MODULES,
              LORMA_RANK_ATTN, LORMA_RANK_FFN, LORMA_ALPHA, LORMA_DROPOUT)

# Build full model
model = RRClassifierArch2(backbone, NUM_LABELS, num_domains=2, dropout=0.1).to(device)

print('\nFinal parameter count:')
count_trainable(model)

## 9. Dataset

In [ ]:
class RRDocumentDatasetArch2(Dataset):
    """
    Returns full document tensors with domain label.
    Uses context window for sentence encoding.
    """
    def __init__(self, docs, tokenizer, label2id, max_seq_len, context_window=3,
                 domain_id=0, is_test=False):
        self.items = []
        self.domain_id = domain_id
        pad_id = tokenizer.stoi.get('<pad>', 0)
        for doc in docs:
            sentences = doc['text']
            labels    = doc.get('labels', [None]*len(sentences))
            doc_id    = doc['id']
            all_ids, all_masks, all_lbls = [], [], []
            for i, (sent, lbl) in enumerate(zip(sentences, labels)):
                start = max(0, i-context_window)
                end   = min(len(sentences), i+context_window+1)
                ctx = ' </s> '.join(sentences[start:i]) + ' </s> ' + sent
                ctx += ' </s> ' + ' </s> '.join(sentences[i+1:end])
                ids = tokenizer.encode(ctx, max_len=max_seq_len)
                pad_len = max_seq_len - len(ids)
                mask = [1]*len(ids) + [0]*pad_len
                ids  = ids + [pad_id]*pad_len
                all_ids.append(ids); all_masks.append(mask)
                all_lbls.append(label2id.get(lbl, 0) if (not is_test and lbl) else -1)
            self.items.append({
                'doc_id': doc_id,
                'input_ids': torch.tensor(all_ids, dtype=torch.long),
                'attention_mask': torch.tensor(all_masks, dtype=torch.long),
                'labels': torch.tensor(all_lbls, dtype=torch.long),
                'domain_id': domain_id
            })

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

collate_fn = lambda b: b

# CL=0, IT=1
cl_tr = RRDocumentDatasetArch2(cl_train, tokenizer, LABEL2ID, MAX_SEQ_LEN, 3, domain_id=0)
it_tr = RRDocumentDatasetArch2(it_train, tokenizer, LABEL2ID, MAX_SEQ_LEN, 3, domain_id=1)
cl_dv = RRDocumentDatasetArch2(cl_dev,   tokenizer, LABEL2ID, MAX_SEQ_LEN, 3, domain_id=0)
it_dv = RRDocumentDatasetArch2(it_dev,   tokenizer, LABEL2ID, MAX_SEQ_LEN, 3, domain_id=1)
cl_te = RRDocumentDatasetArch2(cl_test,  tokenizer, LABEL2ID, MAX_SEQ_LEN, 3, domain_id=0, is_test=True)
it_te = RRDocumentDatasetArch2(it_test,  tokenizer, LABEL2ID, MAX_SEQ_LEN, 3, domain_id=1, is_test=True)

from torch.utils.data import ConcatDataset
train_ds = ConcatDataset([cl_tr, it_tr])
dev_ds   = ConcatDataset([cl_dv, it_dv])
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_ds,   batch_size=1, shuffle=False, collate_fn=collate_fn)
print(f'Train={len(train_ds)}, Dev={len(dev_ds)}')

## 10. Training

In [ ]:
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = OneCycleLR(
    optimizer, max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader), epochs=EPOCHS,
    pct_start=0.1, anneal_strategy='cos'
)

def evaluate(model, loader):
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for batch in loader:
            doc = batch[0]
            ids  = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            lbls = doc['labels'].to(device)
            dom  = doc['domain_id']
            logits = model(ids, mask, dom)
            preds  = logits.argmax(-1).cpu().tolist()
            true   = lbls.cpu().tolist()
            all_p.extend(preds); all_l.extend(true)
    vm = [i for i, l in enumerate(all_l) if l >= 0]
    all_p = [all_p[i] for i in vm]; all_l = [all_l[i] for i in vm]
    return (f1_score(all_l, all_p, average='macro', zero_division=0),
            precision_score(all_l, all_p, average='macro', zero_division=0),
            recall_score(all_l, all_p, average='macro', zero_division=0))

best_f1 = 0.0
best_ckpt = os.path.join(OUTPUT_DIR, 'best_lorma_arch2.pt')
print('Starting LORMA fine-tuning...')

for epoch in range(EPOCHS):
    model.train()
    ep_loss, steps = 0.0, 0
    for batch in train_loader:
        doc  = batch[0]
        ids  = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        lbls = doc['labels'].to(device)
        dom  = doc['domain_id']

        logits = model(ids, mask, dom)
        vm = lbls >= 0
        if vm.sum() == 0: continue

        loss = F.cross_entropy(logits[vm], lbls[vm], weight=class_weights)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        ep_loss += loss.item(); steps += 1

    f1, p, r = evaluate(model, dev_loader)
    print(f'Epoch {epoch+1:2d} | Loss={ep_loss/max(1,steps):.4f} | '
          f'Dev Macro-F1={f1:.4f} P={p:.4f} R={r:.4f}')
    if f1 > best_f1:
        best_f1 = f1
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch+1, 'f1': f1}, best_ckpt)
        print(f'  -> Saved best (F1={f1:.4f})')

print(f'\nBest Dev Macro-F1: {best_f1:.4f}')

## 11. Evaluation & Classification Report

In [ ]:
ckpt = torch.load(best_ckpt, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_p, all_l = [], []
with torch.no_grad():
    for batch in dev_loader:
        doc  = batch[0]
        ids  = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        lbls = doc['labels'].to(device)
        dom  = doc['domain_id']
        logits = model(ids, mask, dom)
        all_p.extend(logits.argmax(-1).cpu().tolist())
        all_l.extend(lbls.cpu().tolist())

vm = [i for i, l in enumerate(all_l) if l >= 0]
all_p = [all_p[i] for i in vm]; all_l = [all_l[i] for i in vm]
print(classification_report(all_l, all_p, target_names=LABELS, zero_division=0))
print(f'Macro-F1: {f1_score(all_l, all_p, average="macro", zero_division=0):.4f}')

## 12. Domain-Specific Results

In [ ]:
def eval_domain_ds(model, ds, device):
    loader = DataLoader(ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
    all_p, all_l = [], []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            doc = batch[0]
            ids  = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            lbls = doc['labels'].to(device)
            dom  = doc['domain_id']
            logits = model(ids, mask, dom)
            all_p.extend(logits.argmax(-1).cpu().tolist())
            all_l.extend(lbls.cpu().tolist())
    vm = [i for i, l in enumerate(all_l) if l >= 0]
    ap = [all_p[i] for i in vm]; al = [all_l[i] for i in vm]
    return f1_score(al, ap, average='macro', zero_division=0), ap, al

cl_f1, _, _ = eval_domain_ds(model, cl_dv, device)
it_f1, _, _ = eval_domain_ds(model, it_dv, device)
print(f'CL Dev Macro-F1: {cl_f1:.4f}')
print(f'IT Dev Macro-F1: {it_f1:.4f}')

## 13. Generate Test Predictions

In [ ]:
def gen_preds(model, test_ds, device, id2label):
    loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
    preds = {}
    model.eval()
    with torch.no_grad():
        for batch in loader:
            doc = batch[0]
            ids  = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            dom  = doc['domain_id']
            logits = model(ids, mask, dom)
            pred_ids = logits.argmax(-1).cpu().tolist()
            preds[doc['doc_id']] = [id2label[p] for p in pred_ids]
    return preds

cl_preds = gen_preds(model, cl_te, device, ID2LABEL)
it_preds = gen_preds(model, it_te, device, ID2LABEL)
all_preds = {**cl_preds, **it_preds}

# Sanity check
errors = sum(1 for doc in cl_test+it_test if len(all_preds.get(doc['id'],[])) != len(doc['text']))
print(f'Prediction sanity: {errors} errors (should be 0)')

pred_path = os.path.join(OUTPUT_DIR, 'predictions.json')
with open(pred_path, 'w') as f: json.dump(all_preds, f, indent=2)
zip_path = '/kaggle/working/submission_exp2_lorma_arch2.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, 'predictions.json')
print(f'Submission: {zip_path}')

## 14. Summary

| Experiment | Model | Technique | Key Advantage |
|---|---|---|---|
| 2 | Arch 2 (RoPE+SwiGLU+RMSNorm, ~105M) | **LORMA** (attn r=16 + FFN r=8) | Adapts both attention and FFN; rank-stabilized scaling; domain tokens |

### LORMA vs LoRA Comparison
- **LoRA** (Exp 1): Only adapts Q/K/V/O attention projections
- **LORMA** (Exp 2): Adapts attention AND gate_proj/up_proj/down_proj in SwiGLU FFN
- LORMA uses a **learned per-layer scalar** `s` for adaptive scaling
- LORMA uses **rank-stabilized** denominator `sqrt(r)` instead of `r`
- Total trainable params ~2.5× more than LoRA but still <5% of total